In [ ]:
# =============================================================================
# Imports
# =============================================================================
 
import numpy as np
import os

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score
 
import tensorflow as tf
import tensorflow_model_optimization as tfmot

keras = tf.keras
layers = tf.keras.layers

In [ ]:
def get_class_weights(y_train):
    """
    Compute inverse-frequency class weights.
 
    The MIT-BIH dataset is heavily imbalanced:
    Normal (N) beats account for ~86 % of all beats.
 
    Passing class_weight to model.fit() increases the loss contribution
    of minority classes (S, V, F, Q), equivalent to oversampling them.
    The paper uses this instead of SMOTE or manual oversampling.
 
    Returns a dict  {class_index: weight}  ready for Keras.
    """
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    return dict(zip(classes.tolist(), weights.tolist()))

In [ ]:
def prune_model(model, target_sparsity=0.5, model_name=None, save_path=None):
    """
    Apply magnitude-based pruning to a model (weights with small magnitude are set to zero)
    
    Args:
        model: The model to prune
        target_sparsity: Target sparsity level (e.g., 0.5 = 50%)
        model_name: Name of the model for saving
        save_path: Directory path to save the pruned model
    
    return: 
       final_model: the final pruned model
       history_phase1: training history during gradual pruning
       history_phase2: training history during constant sparsity fine-tuning
    """
    # Convert labels to one-hot encoding
    y_tr_oh = keras.utils.to_categorical(y_tr, 5)
    y_vl_oh = keras.utils.to_categorical(y_vl, 5)
    y_te_oh = keras.utils.to_categorical(y_te, 5)

    # Get the accuracy of the original model before pruning
    _, original_accuracy = model.evaluate({'signal': Xe_te, 'rr_intervals': Xr_te}, y_te_oh, verbose=0)
    
    # ==================== PHASE 1: GRADUAL PRUNING ====================
    print(f"\n{'='*60}")
    print(f"PHASE 1: Gradual Pruning to {target_sparsity*100:.0f}% Sparsity")
    print(f"{'='*60}")
    
    # Define pruning parameters with polynomial decay schedule
    # This gradually increases sparsity from 0% to target_sparsity over 10 epochs
    pruning_params = {
        'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
            initial_sparsity=0.0,
            final_sparsity=target_sparsity, # Aiming for  50% sparsity (50% of weights become zero), if target_sparsity = 0.5
            begin_step=0,
            end_step=np.ceil(len(Xe_tr) / 256).astype(np.int32) * 10 # Prune over 10 epochs
        )
    }
    
    # Wrap the model with pruning wrappers (adds masks and thresholds to each layer)
    model_for_pruning = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

    # Re-compile the model (Crucial step)
    model_for_pruning.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3), # Lower LR for fine-tuning
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Pruning callback: updates the pruning step counter and applies masks
    callbacks = [
        tfmot.sparsity.keras.UpdatePruningStep(),
        # Recommended: Add a log directory to view sparsity in Tensorboard
        #tfmot.sparsity.keras.PruningSummaries(log_dir='./pruning_logs'),
    ]
    
    history_phase1 = model_for_pruning.fit( 
        x={'signal': Xe_tr, 'rr_intervals': Xr_tr},
        y=y_tr_oh,
        validation_data=({'signal': Xe_vl, 'rr_intervals': Xr_vl}, y_vl_oh),
        epochs=10,
        batch_size=256,
        class_weight=cw,
        callbacks=callbacks
    )

    # Get accuracy of the pruned model befor fine-tuning
    _, phase1_accuracy = model_for_pruning.evaluate({'signal': Xe_te, 'rr_intervals': Xr_te}, y_te_oh, verbose=0)
    
    # Strip pruning wrappers to get a clean model with permanently pruned weights
    # This removes masks, thresholds, and wrapper layers
    stripped_model = tfmot.sparsity.keras.strip_pruning(model_for_pruning)


    # ==================== PHASE 2: CONSTANT SPARSITY FINE-TUNING ====================
    print(f"\n{'='*60}")
    print(f"PHASE 2: Fine-tuning at Constant {target_sparsity*100:.0f}% Sparsity")
    print(f"{'='*60}")

    # Re-wrap with ConstantSparsity to lock the zeros in place
    # This prevents pruned weights from becoming non-zero during fine-tuning
    ft_params = {
        'pruning_schedule': tfmot.sparsity.keras.ConstantSparsity(
            target_sparsity=target_sparsity, # Keep sparsity constant at target
            begin_step=0                     # Apply mask immediately
        )
    }

    model_for_ft = tfmot.sparsity.keras.prune_low_magnitude(stripped_model, **ft_params)
    
    # Recompile with even lower learning rate for stable fine-tuning
    model_for_ft.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Fine-tuning callbacks with early stopping
    callbacks_for_ft = [
        tfmot.sparsity.keras.UpdatePruningStep(),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ]
    
    history_phase2 = model_for_ft.fit(
        x={'signal': Xe_tr, 'rr_intervals': Xr_tr},
        y=y_tr_oh,
        validation_data=({'signal': Xe_vl, 'rr_intervals': Xr_vl}, y_vl_oh),
        epochs=15,
        batch_size=256,
        class_weight=cw,
        callbacks=callbacks_for_ft,
        verbose=1
    )


    # ==================== FINAL MODEL EXPORT ====================
    print(f"\n{'='*60}")
    print("FINAL MODEL EXPORT")
    print(f"{'='*60}")

    # Strip pruning wrappers one final time to get clean model for deployment
    final_model = tfmot.sparsity.keras.strip_pruning(model_for_ft)


    # Save the final pruned model if save_path and model_name are provided
    if save_path is not None and model_name is not None:
        # Create directory if it doesn't exist
        os.makedirs(save_path, exist_ok=True)
        
        # Create descriptive filename with sparsity percentage
        filename = f'pruned_{model_name}_sp{int(target_sparsity*100)}.keras'
        saving_path = os.path.join(save_path, filename)
        
        # Save the model
        final_model.save(saving_path)
        print(f"Model saved to: {saving_path}")
    else:
        print("Model not saved: save_path or model_name not provided")
    
    # ==================== SPARSITY VERIFICATION ====================
    print("\n--- Final Sparsity Verification ---")
    for layer in final_model.layers:
        if isinstance(layer, (keras.layers.Conv1D, keras.layers.Dense, keras.layers.Conv2D)):
            weights = layer.get_weights()[0]
            sparsity = np.mean(weights == 0)
            print(f"Layer {layer.name:15} | Sparsity: {sparsity:.2%}")
    
    # ==================== MODEL EVALUATION ====================
    y_pred = np.argmax(final_model.predict({'signal': Xe_te, 'rr_intervals': Xr_te}), axis=1)
    phase2_accuracy = accuracy_score(y_te, y_pred)
    print(f"\nFinal pruned {model_name} report:")
    print(classification_report(y_te, y_pred, target_names=['N', 'S', 'V', 'F', 'Q'], digits=4))
    
    # ==================== ACCURACY SUMMARY ====================
    print(f"Accuracy summary of the pruned model {model_name} with {target_sparsity*100:.0f}% Sparsity : ") 
    
    print(f"Original Model:       {original_accuracy*100:.2f}%")
    print(f"Phase 1 (Pruned):     {phase1_accuracy*100:.2f}%  ↓ {(original_accuracy-phase1_accuracy)*100:.2f}%")
    print(f"Phase 2 (Fine-tuned): {phase2_accuracy*100:.2f}%)  ↑ {(phase2_accuracy-phase1_accuracy)*100:.2f} %")
    print(f"Final vs Original (lost accuracy):   {(phase2_accuracy - original_accuracy)*100:+.2f}%")


### Main usage

In [ ]:
# Load saved ECG MIT-BIH Arrhythmia dataset
data = np.load("processed_mit_bih_arrhythmia_dataset.npz")

# Extract the arrays back into variables
Xe_tr, Xr_tr, y_tr = data['Xe_tr'], data['Xr_tr'], data['y_tr']
Xe_vl, Xr_vl, y_vl = data['Xe_vl'], data['Xr_vl'], data['y_vl']
Xe_te, Xr_te, y_te = data['Xe_te'], data['Xr_te'], data['y_te']

# Get class weights
cw = get_class_weights(y_tr)

print("Data loaded successfully.")
print(f"Train shapes: {Xe_tr.shape}, {Xr_tr.shape}")

In [ ]:
# Get class weights
cw = get_class_weights(y_tr)
print("  Class weights   :", {k: f"{v:.3f}" for k, v in cw.items()})

In [ ]:
# Load and prune the model
model_name = 'conv1d_2d_t' # There are 4 models in total (con1d_t, con1d_t_fir, con1d_2d_t and con1d_2d_t_fir)
model_path = f'stft_cnn_ecg_classifier_models/{model_name}_model.keras'
model_to_prune = keras.models.load_model(model_path)

save_path = 'ecg_classifier_pruned_models'
target_sparsity_list = [0.3, 0.4, 0.5, 0.6, 0.7]
for target_sparsity in target_sparsity_list:
    prune_model(model=model_to_prune, target_sparsity=target_sparsity, model_name=model_name, save_path=save_path)

In [ ]:
# Model Quantization

In [ ]:
def quantize_and_save(model, export_dir, model_name, quantization_type='int8', representative_data=None):
    """
    Quantizes a Keras model to TFLite format.
    Types: 'float16', 'int16' (dynamic), 'int8' (dynamic), 'full_int8'
    """
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    if quantization_type == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quantization_type == 'int16':
        # Integer quantization with int16 activations and int8 weights
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.int16]
        
    elif quantization_type == 'int8':
        # Dynamic range quantization (weights to int8, activations float)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        
    elif quantization_type == 'full_int8':
        # Full integer quantization (requires representative dataset)
        if representative_data is None:
            raise ValueError("Full integer quantization requires representative_data.")
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_data
        # Ensure fallback to float if an op isn't supported in int8
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    
    # Save the model
    save_path = f"{export_dir}/{model_name}_{quantization_type}.tflite"
    with open(save_path, "wb") as f:
        f.write(tflite_model)
    
    print(f"Quantized model saved to: {save_path}")

def representative_data_gen():
    samples_per_class = 20
    unique_classes = np.unique(y_tr)
    representative_indices = []

    for cls in unique_classes:
        indices = np.where(y_tr == cls)[0]
        selected = np.random.choice(indices, samples_per_class, replace=False)
        representative_indices.extend(selected)

    np.random.shuffle(representative_indices)

    for idx in representative_indices:
        signal = Xe_tr[idx].astype(np.float32)   # (64, 1)
        rr = Xr_tr[idx].astype(np.float32)       # (2,)

        # KEY FIX → force 4D (Conv1D → Conv2D internally)
        signal = np.expand_dims(signal, axis=0)   # (1, 64, 1)
        signal = np.expand_dims(signal, axis=2)   # (1, 64, 1, 1)

        # RR stays 2D
        rr = np.expand_dims(rr, axis=0)           # (1, 2)

        yield [signal, rr]

In [ ]:
# Load the model to quantize
model_to_quantize = tf.keras.models.load_model('stft_cnn_ecg_classifier_models/pruned_conv1d_t_model_sp50.keras')
export_dir = 'ecg_classifier_quantized_models'
model_name = 'pruned_conv1d_t_model_sp50'
quantization_type = 'int8' # It could be float16, int16, int8 or full_int8 quantization


quantize_and_save(model=model_to_quantize, export_dir=export_dir, model_name=model_name, quantization_type=quantization_type)

In [ ]:
def evaluate_tflite_model(tflite_path, Xe_test, Xr_test, y_test):

    # Get model size first
    size_bytes = os.path.getsize(tflite_path)
    if size_bytes < 1024 * 1024:
        size_str = f"{size_bytes / 1024:.2f} KB"
    else:
        size_str = f"{size_bytes / (1024 * 1024):.2f} MB"
        
    # Load the TFLite model and allocate tensors
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    # Get input and output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    y_pred = []

    print(f"Evaluating {tflite_path}...")
    print(f"Model size: {size_str}")
    
    for i in range(len(Xe_test)):
        # Prepare inputs (must match the order in input_details)
        # Check input_details[0]['name'] if you aren't sure of the order
        interpreter.set_tensor(input_details[1]['index'], Xe_test[i:i+1].astype(np.float32))
        interpreter.set_tensor(input_details[0]['index'], Xr_test[i:i+1].astype(np.float32))

        # Run inference
        interpreter.invoke()

        # Get the result
        output_data = interpreter.get_tensor(output_details[0]['index'])
        y_pred.append(np.argmax(output_data))

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    print(f"\nAccuracy: {acc*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=['N', 'S', 'V', 'F', 'Q']))

In [ ]:
# Evaluate any quantized model
tflite_path = './ecg_classifier_quantized_models/pruned_conv1d_t_model_sp50_int8.tflite'
evaluate_tflite_model(tflite_path=tflite_path, Xe_test=Xe_te, Xr_test=Xr_te, y_test=y_te)